In [49]:
import sys
sys.path.insert(0, 'src')
import importlib
import run_pathway_materials
importlib.reload(run_pathway_materials)
from run_pathway_materials import run_pathway_materials
import pandas as pd
import plotly.express as px
import os
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Run Pathway Materials sans aucune contrainte -> Resultat Pathway

In [17]:

results_materials = run_pathway_materials('my_first_run_materials', verbose=True, skip_if_exists=True)

#results_materials['F_new']                    # capacités nouvelles [Phases, Technologies]
#results_materials['Material_content_year']    # demande matériau annualisée [Years, Technologies, Materials]
#results_materials['Recycled_material']

[run_pathway_materials] my_first_run_materials — pkl exists, loading from disk.


In [18]:
material_content = (results_materials['Material_content_year']['Material_content_year']
                     .groupby(['Technologies', 'Materials']).sum() * 5)

In [35]:
periods = ['2020_2025', '2025_2030', '2030_2035', '2035_2040', '2040_2045', '2045_2050']
years = ['2025', '2030', '2035', '2040', '2045', '2050']
elec_keywords = ['PV_', 'WIND_', 'HYDRO', 'NUCLEAR', 'CCGT', 'COAL_', 'OCGT_', 'TIDAL', 'GEOTHERMAL', 'AFC', 'PAFC', 'PEMFC', 'SOFC', 'WAVE']
years_order = ['YEAR_2020','YEAR_2025','YEAR_2030','YEAR_2035','YEAR_2040','YEAR_2045','YEAR_2050']


In [31]:
def def_elec_positive(results_materials):
    technologies = [t for t in results_materials['F_new'].loc['2020_2025'].index
                    if any(results_materials['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]


    elec_techs = [t for t in results_materials['F_new'].loc['2020_2025'].index 
                if any(kw in t for kw in elec_keywords) and not t.startswith(('COAL_GAS', 'HYDRO_STORAGE', 'UNMINEABLE_COAL_SEAM'))]

    elec_techs_positive = [t for t in elec_techs
                        if any(results_materials['F_new'].loc[period].loc[t].squeeze() > 0 for period in periods)]
    
    return elec_techs_positive

elec_techs_positive = def_elec_positive(results_materials)


In [33]:
def plot_elec_new_positive(results_materials, elec_techs_positive):

    df_plot = pd.DataFrame(
        {period: results_materials['F_new'].loc[period].loc[elec_techs_positive].squeeze() for period in periods},
        index=elec_techs_positive
    )

    df_melted = df_plot.T.reset_index().rename(columns={'index': 'Période'}).melt(
        id_vars='Période', var_name='Technologies', value_name='Capacité'
    )

    fig = px.bar(df_melted, x='Période', y='Capacité', color='Technologies', barmode='stack')
    fig.update_layout(xaxis_title='Période', yaxis_title='Capacité [GW]')
    fig.show()

In [39]:
def plot_elec_mult_positive(results_materials, elec_techs_positive):

    f_mult = results_materials['F_Mult'].reset_index()

    f_mult_elec = f_mult[f_mult['Technologies'].apply(lambda t: any(kw in t for kw in elec_keywords))]
    f_mult_elec = f_mult_elec[f_mult_elec['F_Mult'] > 0]  # enleve les lignes a zero

    fig = px.bar(f_mult_elec, x='Years', y='F_Mult', color='Technologies',
                category_orders={'Years': years_order},
                title='Capacité installée (F_Mult) par technologie et par année [GW]')
    fig.show()

In [42]:
def plot_all_material_demand(results_materials, save_file = False):

    mcy = results_materials['Material_content_year']['Material_content_year']
    demand = mcy.groupby(['Years', 'Materials']).sum().unstack('Materials')  # index=Years, colonnes=Materials
    demand = demand.loc[:, (demand.fillna(0) != 0).any(axis=0)]  # enleve les materiaux a zero partout

    materials = demand.columns.tolist()
    n = len(materials)
    ncols = 6
    nrows = -(-n // ncols)

    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=materials)

    years_x = [int(y.replace('YEAR_', '')) for y in demand.index.tolist()]

    for i, material in enumerate(materials):
        row = i // ncols + 1
        col = i % ncols + 1
        fig.add_trace(
            go.Bar(x=years_x, y=demand[material].values, name=material, showlegend=False),
            row=row, col=col
        )

    fig.update_layout(height=300 * nrows, title='Demande annuelle en matériau (modèle pathway + contraintes)')
    fig.update_yaxes(title_text='[t/an]', col=1)
    fig.update_xaxes(tickmode='array', tickvals=years_x, tickangle=45)
    fig.show()

    if save_file:
        save_dir = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/EPFL/PdM/Plots/Elec_energy_infinite')
        os.makedirs(save_dir, exist_ok=True)

        filepath = os.path.join(save_dir, 'tot_mat_elec_infinite.png')
        fig.write_image(filepath, width=250*ncols, height=300*nrows)

In [34]:
plot_elec_new_positive(results_materials, elec_techs_positive)

In [40]:
plot_elec_mult_positive(results_materials, elec_techs_positive)

In [43]:
plot_all_material_demand(results_materials)

# Run Pathway Materials sans contrainte minium de wind, hydro et pv (plan hydro quebec)

In [29]:
results_materials_relax = run_pathway_materials('my_first_run_materials_relax', verbose=True, hydro_quebec_constraints =False)


Presolve eliminates 0 constraints and 109570 variables.
Adjusted problem:
690416 variables:
	6342 binary variables
	84 nonlinear variables
	683990 linear variables
754344 constraints; 3594116 nonzeros
	7 nonlinear constraints
	754337 linear constraints
	614281 equality constraints
	136620 inequality constraints
	3443 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 690416 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  754344 linear;

AMPL MP final model has 693859 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  748985 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: Intel(R) C

In [44]:
elec_positive_relax = def_elec_positive(results_materials_relax)

In [45]:
plot_elec_new_positive(results_materials_relax, elec_positive_relax)

In [47]:
plot_elec_mult_positive(results_materials_relax, elec_positive_relax)

In [48]:
plot_all_material_demand(results_materials_relax)

# Run Pathway Materials sans contrainte minium de wind, hydro et pv (plan hydro quebec) et avec limite matériau

In [51]:
results_materials_relax_limit = run_pathway_materials('my_first_run_materials_relax_limit', verbose=True, hydro_quebec_constraints =False, materials_limit= True)


Presolve eliminates 0 constraints and 109570 variables.
Adjusted problem:
690416 variables:
	6342 binary variables
	84 nonlinear variables
	683990 linear variables
754344 constraints; 3594116 nonzeros
	7 nonlinear constraints
	754337 linear constraints
	614281 equality constraints
	136620 inequality constraints
	3443 range constraints
2 objectives, all linear; 2 nonzeros.

Gurobi 13.0.2:   pre:dual = -1
  alg:method = 2
  bar:crossover = 0
  tech:threads = 0
  pre:passes = 3
  bar:convtol = 9.9999999999999995e-07
  pre:solve = -1
  iis:find = 1
Set parameter LogToConsole to value 1
  tech:outlev = 1

AMPL MP initial flat model has 690416 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  754344 linear;

AMPL MP final model has 693859 variables (0 integer, 6342 binary);
Objectives: 1 linear; 
Constraints:  748987 linear;


Set parameter InfUnbdInfo to value 1
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[x86] - Darwin 23.5.0 23F79)

CPU model: Intel(R) C

In [52]:
elec_positive_relax_limit = def_elec_positive(results_materials_relax_limit)

In [53]:
plot_elec_new_positive(results_materials_relax_limit, elec_positive_relax_limit)

In [54]:
plot_elec_mult_positive(results_materials_relax_limit, elec_positive_relax_limit)

In [55]:
plot_all_material_demand(results_materials_relax_limit)